In [ ]:
import pandas as pd
import json

from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

# import logging
# logging.basicConfig(
#     level=logging.DEBUG,
#     format="%(asctime)s %(levelname)s %(name)s: %(message)s",
#     handlers=[
#         logging.FileHandler("curation.log"),
#         logging.StreamHandler(),  # keep console output too
#     ],
#     force=True,
# )

# Download data


In [ ]:
noncurated_path = "../non_curated/h5ad/gasperini_2019_atscale.h5ad"
download_file(
    url="https://exampledata.scverse.org/pertpy/gasperini_2019_atscale.h5ad",
    dest_path=noncurated_path
)

# Initialise the dataset object

In [ ]:
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    noncurated_path=noncurated_path
)

cur_data.load_data()

# OBS slot curation

### Rename gene -> perturbation_name, barcode -> guide_sequence

In [ ]:
cur_data.rename_columns(slot = 'obs',
                        name_dict = {"gene": "perturbation_name",
                                     "barcode": "guide_sequence"})
# add cell barcode
cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.astype(str)
cur_data.adata.obs

### Add cell barcodes to the obs slot

In [ ]:
cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.astype(str)

print(cur_data.adata.obs[['cell_barcode']].head())

### Show unique perturbations

In [ ]:
cur_data.show_unique(slot = 'obs', column = 'perturbation_name')

### Add guide RNA information

In [ ]:
import re

# define regex pattern for splitting
controls_regex_pattern = r"(top_two|second_two|TSS|bassik_mch|pos_control_HBE1_tss_Klann_mosaic|pos_control_HBE1_tss_Klann_mosaic|pos_control_HS2_Klann_mosaic|pos_control_Klannchr1_HBG1_HBG1_tss_both|pos_control_Klannchr1_HS3|pos_control_Klannchr1_HS4|pos_control_Klannchr_HS1|pos_control_KlannHS2g_HS2_A|pos_control_KlannHS2g_HS2_B|pos_control_mosaic_HB_HBE1_tss_A|pos_control_mosaic_HB_HBE1_tss_B|scrambled_\d+|random_\d+)"
# split the gene names based on the regex pattern
gene_split = [re.sub(controls_regex_pattern, r"\1\n", e).split('\n') for e in cur_data.adata.obs['perturbation_name']]
# clean up the split parts
gene_split = [[i.strip('_').replace('_TSS', '') for i in e if i != ''] for e in gene_split]
# join the split parts with '|'
gene_split = ['|'.join(e) for e in gene_split]

cur_data.adata.obs['gene_split'] = gene_split

cur_data.adata.obs['gene_split']

In [ ]:
# download supplementary files on at-scale guide annotations
download_file(
    url="https://ars.els-cdn.com/content/image/1-s2.0-S009286741831554X-mmc2.xlsx",
    dest_path="../supplementary/gasperini_2019_atscale_supp.xlsx"
)
download_file(
    url="https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE120861&format=file&file=GSE120861%5Fgrna%5Fgroups%2Eat%5Fscale%2Etxt%2Egz",
    dest_path="../supplementary/gasperini_2019_atscale_supp.txt.gz"
)
# load guide info
guide_info_df = pd.read_excel("../supplementary/gasperini_2019_atscale_supp.xlsx", sheet_name="S2A_AtScale_library_gRNA.cs", skiprows=0)
guide_info_df2 = pd.read_table("../supplementary/gasperini_2019_atscale_supp.txt.gz", sep='\t', header=None, names=['target', 'guide_sequence'])
# add coordinate info
guide_info_df['start.candidate_enhancer'] = guide_info_df['start.candidate_enhancer'].astype(str).str.replace('.0', '')
guide_info_df['stop.candidate_enhancer'] = guide_info_df['stop.candidate_enhancer'].astype(str).str.replace('.0', '')
guide_info_df['coord'] = (guide_info_df['chr.candidate_enhancer']+ ':' +
                              guide_info_df['start.candidate_enhancer'] + '-' +
                              guide_info_df['stop.candidate_enhancer'])
guide_info_df.loc[guide_info_df['coord'] == 'nan:nan-nan', 'coord'] = ''
guide_info_df = guide_info_df[['Spacer', 'coord']]#.fillna('')#.dropna()
# remove _TSS from target names
guide_info_df2['target'] = guide_info_df2['target'].str.replace('_TSS', '')
guide_info_df2 = guide_info_df2.merge(guide_info_df, left_on='guide_sequence', right_on='Spacer', how='left').drop(columns=['Spacer'])

guide_info_df2


In [ ]:
# get gene ont
gene_ont = cur_data.gene_ont.dropna(subset='synonym')
# remove unnecessary mappings based on chr
gene_ont = gene_ont[gene_ont['chromosome_name'].isin(
    [str(i) for i in range(1, 23)] + ["X", "Y", "MT"]
)]

# create a mapping dict name:coord
enh_mapping_dict = (
    guide_info_df2[guide_info_df2['target'].str.startswith('chr')][['target', 'coord']]
    .drop_duplicates()
    .set_index('target')['coord']
    .to_dict()
)

# init enhancer mapping df
enh_mapping_df = pd.DataFrame(columns=gene_ont.columns)
enh_mapping_df['synonym'] = enh_mapping_dict.keys()
enh_mapping_df['ensembl_gene_id'] = enh_mapping_dict.keys()
enh_mapping_df['ensembl_gene_id'] = enh_mapping_df['ensembl_gene_id'].str.replace('chr', 'enh_chr_')
enh_mapping_df['gene_symbol'] = enh_mapping_df['ensembl_gene_id']
enh_mapping_df['gene_coord'] = enh_mapping_dict.values()
enh_mapping_df['chromosome_name'] = enh_mapping_df['gene_coord'].str.split(':').str[0].str.replace('chr', '')
enh_mapping_df['biotype'] = 'enhancer'
enh_mapping_df['synonym_type'] = 'symbol_syn'
enh_mapping_df['description'] = 'enhancer'

# concat to gene_ont df
gene_ont = pd.concat([gene_ont, enh_mapping_df], axis=0, ignore_index=True)

# replace cur_data.gene_ont with the updated one
cur_data.gene_ont = gene_ont

gene_ont.tail()

In [ ]:
# replace controls in gene_split with mappable ones
cur_data.adata.obs['gene_split'] = (
    cur_data.adata.obs['gene_split']
        .replace(
        {r'random_\d+': 'control_genedesert',
         r'scrambled_\d+': 'control_nontargeting',
         r'pos_control_\w+': 'control_positive'
         },
        regex=True
    )
)


In [ ]:
cur_data.adata.obs

### Standardise perturbation targets

In [ ]:
cur_data.standardize_genes(
    slot='obs',
    input_column='gene_split',
    input_column_type='gene_symbol',
    multiple_entries=True,
    multiple_entries_sep='|'
)

In [ ]:
# clean the mapped names ensg and symbol columns
controls_regex_pattern = r"(_top_two|_second_two)"

cur_data.adata.obs['perturbed_target_ensg'] = cur_data.adata.obs['perturbed_target_ensg'].str.replace(controls_regex_pattern, '', regex=True)
cur_data.adata.obs['perturbed_target_symbol'] = cur_data.adata.obs['perturbed_target_symbol'].str.replace(controls_regex_pattern.upper(), '', regex=True)


### Add `perturbed_target_number` column

In [ ]:
cur_data.count_entries(
    slot='obs',
    input_column='perturbed_target_symbol',
    count_column_name='perturbed_target_number',
    sep='|'
)

### Encode chromosomes as integers

In [ ]:
cur_data.chromosome_encoding()

### Add replicate information

In [ ]:
cur_data.adata.obs = cur_data.adata.obs.rename(
    columns={
        'prep_batch': 'technical_replicate'
    }
)

### Add guide sequence information

In [ ]:
cur_data.adata.obs['guide_sequence'] = cur_data.adata.obs['guide_sequence'].str.replace('_', '|')

### Add metadata

In [ ]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        #----- dataset -----#
        "dataset_id": cur_data.dataset_id,
        #----- sample -----#
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        #----- perturbation type -----#
        "perturbation_type_label": "CRISPRi",
        "perturbation_type_id": None,
        #----- data modality -----#
        "data_modality": "Perturb-seq", # different from "method_name_label"; more general term - choice of CRISPR, MAVE and Perturb-seq
        #----- significance -----#
        "significant": None,
        "significance_criteria": None,
        #----- score interpretation -----#
        "score_interpretation": None,
        #----- treatment -----#
        "treatment_label": None,
        "treatment_id": None,
        #----- replicate -----#
        # "technical_replicate": None,
        "biological_replicate": None,
        #----- model system -----#
        "model_system_label": "cell_line",
        "model_system_id": None,
        #----- tissue -----#
        "tissue": "blood",
        #----- cell line -----#
        "cell_line_label": "K 562 cell",
        # "cell_line_id": None,
        #----- cell type -----#
        "cell_type_label": "lymphoblast",
        # "cell_type_id": ,
        #----- disease -----#
        "disease_label": "chronic myelogenous leukemia, BCR-ABL1 positive",
        "disease_id": "MONDO:0011996",
        #----- timepoint -----#
        "timepoint": "P10DT0H0M0S",
        #----- species -----#
        "species": "Homo sapiens",
        #----- sex -----#
        "sex_label": "female",
        "sex_id": None,
        #----- developmental stage -----#
        "developmental_stage_label": "adult",
        "developmental_stage_id": None,
        #----- study metadata -----#
        "study_title": "A Genome-wide Framework for Mapping Gene Regulation via Cellular Genetic Screens",
        "study_uri": "https://doi.org/10.1016/j.cell.2018.11.029",
        "study_year": 2019,
        #----- authors -----#
        "first_author": "Molly Gasperini",
        "last_author": "Jay Shendure",
        #----- experiment metadata -----#
        "experiment_title": "Perturb-seq CRISPRi screen in K562 cells to explore the targets of over 5,779 candidate enhancers.",
        "experiment_summary": """
            Several methods were used to select a set of 5,799 candidate enhancers. GuideRNAs targeting these candidate enhancers, as well a selection of non-targeting , gene-desert and positive controls (targeting globin TSS and enhancers + 381 gene TSSs) were cloned into a CRISPRi-optimised CROP-seq vector. After producing the lentiviral library, K562 cells stably expressing the dCas9-BFP-KRAB were transduced with the library at A very high MOI=~28 (median 28 ± 15.3 gRNAs identified per cell) and cultured for 10 days, at which point the cells were harvested for sequencing using Illumina NovaSeq 6000.
        """,
        #----- number of perturbed targets/samples -----#
        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_coord'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],
        #----- library generation type -----#
        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",
        #----- library generation method -----#
        "library_generation_method_id": "EFO:0022895",
        "library_generation_method_label": "dCas9-KRAB",
        #----- enzyme and library delivery method -----#
        "enzyme_delivery_method_id": None,
        "enzyme_delivery_method_label": "lentivirus transduction",

        "library_delivery_method_id": None,
        "library_delivery_method_label": "lentivirus transduction",
        #----- enzyme and library integration state -----#
        "enzyme_integration_state_id": None,
        "enzyme_integration_state_label": "random locus integration",

        "library_integration_state_id": None,
        "library_integration_state_label": "random locus integration",
        #----- enzyme and library expression control -----#
        "enzyme_expression_control_id": None,
        "enzyme_expression_control_label": "constitutive transgene expression",

        "library_expression_control_id": None,
        "library_expression_control_label": "constitutive transgene expression",
        #----- library name and URI and manufacturer -----#
        "library_name": "custom",
        "library_uri": None,
        "library_manufacturer": "Shendure lab",
        #----- library format -----#
        "library_format_id": None,
        "library_format_label": "pooled",
        #----- library scope -----#
        "library_scope_id": None,
        "library_scope_label": "focused",
        #----- library perturbation type -----#
        "library_perturbation_type_id": None,
        "library_perturbation_type_label": "inhibition",
        #----- library additional metadata -----#
        "library_lentiviral_generation": "3",
        "library_grnas_per_target": "2",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()), # for CRISPR/Perturb-seq
        "library_total_variants": None, # for MAVE
        #----- readout dimensionality -----#
        "readout_dimensionality_id": None,
        "readout_dimensionality_label": "high-dimensional assay",
        #---- readout type -----#
        "readout_type_id": None,
        "readout_type_label": "transcriptomic",
        #----- readout technology -----#
        "readout_technology_id": None,
        "readout_technology_label": "single-cell rna-seq",
        #----- method -----#
        "method_name_id": None,
        "method_name_label": "Perturb-seq", # different from "data_modality"; more specific term - specific name of the technique
        "method_uri": None,
        #----- sequencing library kit -----#
        "sequencing_library_kit_id": None,
        "sequencing_library_kit_label": "10x Genomics Single Cell 3-prime v2",
        #----- sequencing platform -----#
        "sequencing_platform_id": None,
        "sequencing_platform_label": "Illumina NovaSeq 6000",
        #----- sequencing strategy -----#
        "sequencing_strategy_id": None,
        "sequencing_strategy_label": "barcode sequencing",
        #----- software used for counts-----#
        "software_counts_id": None,
        "software_counts_label": "CellRanger",
        #----- software used for analysis -----#
        "software_analysis_id": None,
        "software_analysis_label": "Seurat",
        #----- reference genome -----#
        "reference_genome_id": None,
        "reference_genome_label": "GRCh37",
        #----- license -----#
        "license_label": "free to use license",
        "license_id": "SWO:1000061",
        #----- external datasets -----#
        "associated_datasets": json.dumps([
            {
                "dataset_accession": "gasperini_2019_atscale.h5ad",
                "dataset_uri": "https://exampledata.scverse.org/pertpy/gasperini_2019_atscale.h5ad",
                "dataset_description": "Raw counts - .h5ad file from pertpy",
                "dataset_file_name": "gasperini_2019_atscale.h5ad",
            },
            {
                "dataset_accession": "GSE120861",
                "dataset_uri": "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE120861",
                "dataset_description": "Raw counts - GEO entry",
                "dataset_file_name": "GSE120861_at_scale_screen.*",
            }
        ])
    }
)

### Curate tissue information


In [ ]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

### Curate cell type information


In [ ]:
cur_data.standardize_ontology(
    input_column='cell_type_label',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

### Curate cell line information

In [ ]:
cur_data.standardize_ontology(
    input_column='cell_line_label',
    column_type='term_name',
    ontology_type='cell_line',
    overwrite=True
)

### Curate disease information

In [ ]:
cur_data.standardize_ontology(
    input_column='disease_label',
    column_type='term_name',
    ontology_type='disease',
    overwrite=True
)

### Match schema column order

In [ ]:
cur_data.match_schema_columns(slot='obs')

### Validate obs metadata

In [ ]:
cur_data.validate_data(slot='obs', verbose=True)

# VAR slot curation

### Standardise genes

In [ ]:
cur_data.adata.var['gene_name'] = cur_data.adata.var.index
cur_data.adata.var

In [ ]:
cur_data.standardize_genes(
    slot="var",
    input_column="ensembl_id",
    input_column_type="ensembl_gene_id",
    remove_version=False,
    multiple_entries=False
)

In [ ]:
# replace missing non-ENSMBL entries in gene-symbols with the originals
cur_data.adata.var.loc[cur_data.adata.var['gene_symbol'].isna() & ~cur_data.adata.var['original_index'].str.startswith('ENSG'), 'gene_symbol'] = cur_data.adata.var['gene_symbol'].fillna(cur_data.adata.var['original_index'])

In [ ]:
cur_data.adata.var

### Validate var metadata

In [ ]:
cur_data.validate_data(slot='var')

# Save the dataset

In [ ]:
cur_data.save_curated_data_h5ad()

In [ ]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

# Upload to BigQuery

In [ ]:
upload_parquet_to_bq(
    parquet_path='../curated/parquet/gasperini_2019_atscale_curated_metadata.parquet',
    bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
    bq_table_name='metadata',
    key_columns=['dataset_id', 'sample_id'],
    verbose=True
)

# Upload to GC Storage

In [ ]:
!gcloud storage cp ../curated/h5ad/gasperini_2019_atscale_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/